# Sector Rotation System — Setup & Backtest

This notebook runs on Google Colab (free tier). It:

1. Clones the repo and installs dependencies
2. Populates the database with 2 years of historical data
3. Force-injects any missing tickers (Fidelity MSCI, IDEQ, OTC ADRs)
4. Detects the current market regime
5. Runs the stock screener and lets you confirm picks
6. Runs the CVaR optimizer using confirmed picks
7. Executes a walk-forward backtest
8. Shows the Executive Summary
9. Lets you review and confirm the final allocation before downloading

**Runtime: ~8–12 minutes on Colab free tier.**

## 1. Clone Repo & Install Dependencies

Clones the repo and installs dependencies.

> Store your GitHub PAT in Colab Secrets as `GITHUB_TOKEN` with `repo` scope.

In [1]:
import os
import importlib
from pathlib import Path

REPO_DIR = '/content/sector-rotation-system'

if not os.path.isdir(REPO_DIR):
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    os.system(f'git clone https://{token}@github.com/bigbathtub101/sector-rotation-system.git {REPO_DIR}')
else:
    print(f'Repo already cloned at {REPO_DIR}, pulling latest...')
    os.system(f'cd {REPO_DIR} && git pull')
    # Force-reload every project module so new code takes effect after git pull
    import sys
    project_modules = [
        'data_feeds', 'regime_detector', 'portfolio_optimizer',
        'stock_screener', 'nlp_sentiment', 'etf_selector', 'monitor',
    ]
    for mod_name in project_modules:
        if mod_name in sys.modules:
            importlib.reload(sys.modules[mod_name])
            print(f'  ~ Reloaded {mod_name}')

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

os.system('pip install -q -r requirements.txt')
print('\nSetup complete.')

Working directory: /content/sector-rotation-system

Setup complete.


## 2. Set API Keys

Sets FRED and SEC EDGAR API keys from Colab Secrets. FRED is optional — the system uses placeholder macro data if not set.

In [2]:
import os
import re

try:
    from google.colab import userdata
    fred_key = userdata.get("FRED_API_KEY")
    if fred_key:
        os.environ["FRED_API_KEY"] = fred_key
except Exception:
    pass

try:
    from google.colab import userdata
    sec_email = userdata.get("SEC_EDGAR_EMAIL")
    if sec_email:
        sec_email = re.sub(r"[^\x20-\x7E]", "", sec_email).strip()
        if sec_email:
            os.environ["SEC_EDGAR_EMAIL"] = sec_email
        else:
            print("WARNING: SEC_EDGAR_EMAIL contained only control characters -- ignoring.")
except Exception:
    pass

fred_key = os.environ.get("FRED_API_KEY", "")
print(f"FRED_API_KEY: {('set (' + fred_key[:4] + '...)') if fred_key else 'NOT SET (will use dummy macro data)'}")
sec_email = os.environ.get("SEC_EDGAR_EMAIL", "")
print(f"SEC_EDGAR_EMAIL: {('set (' + repr(sec_email[:20]) + '...)') if sec_email else 'NOT SET (will use config.yaml default)'}")

FRED_API_KEY: set (a5c3...)
SEC_EDGAR_EMAIL: set ('gabop98@gmail.com'...)


## 3. Load Config & Backfill Historical Data

Downloads 2 years of daily prices for all ETFs and watchlist stocks, macro data from FRED, and recent SEC filings. Incremental on subsequent runs.

In [3]:
import yaml
import sqlite3
import inspect
import importlib
import pandas as pd
import datetime as dt
import gc
from pathlib import Path

with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

DB_PATH = Path(os.getcwd()) / 'rotation_system.db'

def patch_db_path(mod, new_path):
    """Overwrite a module's DB_PATH global AND fix any function
    defaults that captured the old value at import time."""
    old_path = getattr(mod, 'DB_PATH', None)
    mod.DB_PATH = new_path
    for name, obj in inspect.getmembers(mod, inspect.isfunction):
        if obj.__defaults__:
            new_defaults = []
            changed = False
            for d in obj.__defaults__:
                if isinstance(d, Path) and old_path and d == old_path:
                    new_defaults.append(new_path)
                    changed = True
                else:
                    new_defaults.append(d)
            if changed:
                obj.__defaults__ = tuple(new_defaults)
    print(f'  {mod.__name__}.DB_PATH -> {mod.DB_PATH}')

print("Config loaded.")
print(f"  Portfolio total:  ${config['portfolio']['total_value']:,.0f}")
print(f"  Taxable:          ${config['portfolio']['accounts']['taxable']['value']:,.0f}")
print(f"  Roth IRA:         ${config['portfolio']['accounts']['roth_ira']['value']:,.0f}")
print(f"  Sector ETFs:      {len(config['tickers']['sector_etfs'])}")
wl_keys = ['watchlist_biotech', 'watchlist_ai_software', 'watchlist_defense',
           'watchlist_green_materials', 'watchlist_semiconductors',
           'watchlist_energy_transition', 'watchlist_fintech']
wl_count = sum(len(config['tickers'].get(k, [])) for k in wl_keys)
print(f"  Watchlist tickers: {wl_count}")
print(f"  DB path: {DB_PATH}")

# Force cleanup of any stale DB connections from prior cell runs
gc.collect()

if DB_PATH.exists():
    try:
        _test_conn = sqlite3.connect(str(DB_PATH), timeout=30)
        _test_conn.execute("PRAGMA journal_mode=WAL")
        _test_conn.execute("SELECT 1")
        _test_conn.close()
        print(f"\n[OK] DB file verified: {DB_PATH}")
    except sqlite3.OperationalError as e:
        print(f"[!] DB appears locked, removing stale file: {e}")
        DB_PATH.unlink()

import data_feeds
importlib.reload(data_feeds)
patch_db_path(data_feeds, DB_PATH)

conn = data_feeds.init_database(DB_PATH)

PRICE_FRESHNESS_DAYS = 3
MIN_FILINGS_THRESHOLD = 50

# Patch for NameError: name 'lookup_cik' is not defined
if not hasattr(data_feeds, 'lookup_cik'):
    import logging
    _logger = logging.getLogger(__name__)
    def _placeholder_lookup_cik(ticker, cfg, log_warning=True):
        if log_warning:
            _logger.warning("Using placeholder lookup_cik for ticker %s.", ticker)
        return None
    data_feeds.lookup_cik = _placeholder_lookup_cik
    print("[!] Applied temporary patch: data_feeds.lookup_cik placeholder added.")

try:
    # --- Prices ---
    print("\n[>>] Checking price data...")
    latest_date = None
    try:
        row = conn.execute("SELECT MAX(date) FROM prices").fetchone()
        if row and row[0]:
            latest_date = row[0]
    except Exception:
        pass

    today = dt.date.today()
    if latest_date and (today - dt.date.fromisoformat(latest_date)).days <= PRICE_FRESHNESS_DAYS:
        print(f"[OK] Prices are current (latest: {latest_date}). Skipping download.")
    else:
        if latest_date:
            price_start = (dt.date.fromisoformat(latest_date) + dt.timedelta(days=1)).isoformat()
            print(f"[>>] Fetching incremental prices from {price_start}...")
        else:
            price_start = None
            print("[>>] First run -- fetching full 2-year price history...")
        prices = data_feeds.fetch_prices(config, start_date=price_start)
        prices, price_warnings = data_feeds.validate_prices(prices, config)
        data_feeds.store_prices(conn, prices)
        if price_warnings:
            print(f"  [!] {len(price_warnings)} price warnings")
        print(f"[OK] Prices stored: {len(prices)} rows, {int(prices['ticker'].nunique()) if not prices.empty else 0} tickers")

    # --- Macro ---
    print("\n[>>] Fetching macro data from FRED...")
    macro = data_feeds.fetch_macro_data(config)
    macro, macro_warnings = data_feeds.validate_macro(macro)
    data_feeds.store_macro(conn, macro)
    print(f"[OK] Macro stored: {len(macro)} rows, {int(macro['series_id'].nunique()) if not macro.empty else 0} series")

    # --- SEC Filings ---
    print("\n[>>] Checking SEC filings...")
    filing_count = conn.execute("SELECT COUNT(*) FROM filings").fetchone()[0]
    if filing_count > MIN_FILINGS_THRESHOLD:
        print(f"[OK] Filings already populated ({filing_count:,} rows). Skipping.")
    else:
        import time as _time
        max_budget = config.get("sec_edgar", {}).get("max_fetch_time_seconds", 300)
        print(f"[>>] First run -- fetching SEC filings (budget: {max_budget}s)...")
        _t0 = _time.monotonic()
        filings = data_feeds.fetch_all_filings(config)
        _elapsed = _time.monotonic() - _t0
        data_feeds.store_filings(conn, filings)
        with_text = sum(1 for f in filings if f.get("raw_text"))
        print(f"[OK] Filings stored: {len(filings):,} total in {_elapsed:.0f}s ({with_text:,} with text)")

finally:
    conn.close()

# Coverage report
print(f"\n=== Ticker Coverage ===")
if DB_PATH.exists():
    print(f"  DB size: {DB_PATH.stat().st_size:,} bytes")
    conn2 = sqlite3.connect(str(DB_PATH))
    try:
        for table in ['prices', 'macro_data', 'filings', 'signals']:
            try:
                count = pd.read_sql(f'SELECT COUNT(*) AS n FROM {table}', conn2).iloc[0]['n']
                print(f"  {table}: {count:,} rows")
            except Exception as e:
                print(f"  {table}: ERROR - {e}")
        loaded = pd.read_sql('SELECT DISTINCT ticker FROM prices', conn2)['ticker'].tolist()
        all_expected = set()
        for key in config['tickers']:
            if isinstance(config['tickers'][key], list):
                all_expected.update(config['tickers'][key])
        missing = all_expected - set(loaded)
        print(f"\n  Tickers loaded: {len(loaded)} / {len(all_expected)} expected")
        if missing:
            print(f"  Missing: {sorted(missing)}")
    finally:
        conn2.close()

Config loaded.
  Portfolio total:  $235,970
  Taxable:          $98,913
  Roth IRA:         $46,207
  Sector ETFs:      11
  Watchlist tickers: 76
  DB path: /content/sector-rotation-system/rotation_system.db
  data_feeds.DB_PATH -> /content/sector-rotation-system/rotation_system.db

[>>] Checking price data...
[>>] First run -- fetching full 2-year price history...


[*********************100%***********************]  153 of 153 completed


  [!] 1 price warnings
[OK] Prices stored: 76653 rows, 153 tickers

[>>] Fetching macro data from FRED...
[OK] Macro stored: 613 rows, 6 series

[>>] Checking SEC filings...
[>>] First run -- fetching SEC filings (budget: 300s)...


[OK] Filings stored: 228 total in 93s (228 with text)

=== Ticker Coverage ===
  DB size: 26,255,360 bytes
  prices: 76,653 rows
  macro_data: 613 rows
  filings: 226 rows
  signals: 0 rows

  Tickers loaded: 153 / 157 expected
  Missing: ['FCOM', 'FDIS', 'FHLC', 'FIDU']


## 4. Force-Inject Missing Tickers

Force-downloads any tickers missed by the bulk fetch (Fidelity MSCI ETFs, IDEQ, Franklin FTSE, OTC ADRs) and injects them into the price DB. Also manually patches IDEQ holdings from the Lazard Q4 2025 factsheet. **Safe to re-run — uses `INSERT OR REPLACE`.**

In [4]:
# Downloads any tickers that yfinance missed in the bulk fetch (Fidelity MSCI
# ETFs, IDEQ, OTC ADRs) and injects them directly into the price DB.
# Also manually patches IDEQ holdings since yfinance can't scrape them.
# Safe to re-run -- uses INSERT OR REPLACE.
# ============================================================================

import yfinance as yf
import numpy as np

conn_fix = sqlite3.connect(str(DB_PATH))

# ---------------------------------------------------------------------------
# Step 4a: Build complete expected ticker list
# ---------------------------------------------------------------------------
all_tickers_in_config = set()
for key, val in config['tickers'].items():
    if isinstance(val, list):
        all_tickers_in_config.update(val)

# Always include actual holdings regardless of which config key they live in
must_have = [
    'FCOM', 'FDIS', 'FHLC', 'FIDU', 'IDEQ',   # taxable holdings
    'HUBS', 'XYZ',  'BMRN', 'FIS',  'NVDA',    # roth holdings
    'FTEC', 'FNCL', 'FIDU', 'FDIS', 'FSTA',    # Fidelity MSCI (ETF selector candidates)
    'FENY', 'FMAT', 'FUTY', 'FREL', 'FCOM',
    'FLIN', 'FLBR', 'FLJP', 'FLEU', 'FLCH',    # Franklin FTSE (ETF selector candidates)
    'FLKR', 'FLTW',
    'SGOV', 'IEMG',                              # cash + broad EM alternatives
]
all_tickers_in_config.update(must_have)

# Remove non-downloadable items
skip_always = {'^VIX', '^VIX3M'}
all_tickers_in_config -= skip_always

# Check what's already in the DB with sufficient history (>= 100 rows)
loaded_sufficient = set(
    row[0] for row in
    conn_fix.execute(
        "SELECT ticker FROM prices GROUP BY ticker HAVING COUNT(*) >= 100"
    ).fetchall()
)

missing = sorted(all_tickers_in_config - loaded_sufficient)
print(f"[>>] Tickers needed:    {len(all_tickers_in_config)}")
print(f"[OK] Already in DB:     {len(loaded_sufficient)}")
print(f"[!]  Missing / thin:    {len(missing)}")
if missing:
    print(f"     {missing}")

# ---------------------------------------------------------------------------
# Step 4b: Batch download missing tickers
# ---------------------------------------------------------------------------
# Known OTC/illiquid tickers that will always fail on yfinance -- skip silently
expected_failures = {'BAESY', 'RNMBY', 'SAABF', 'SGML', 'IBUY', 'UFO', 'KOMP'}

downloadable = [t for t in missing if t not in expected_failures]

if downloadable:
    print(f"\n[>>] Downloading {len(downloadable)} missing tickers via yfinance...")

    def inject_ticker_prices(conn_inj, ticker, hist_df):
        """Insert a price history DataFrame into the prices table."""
        if hist_df.empty or len(hist_df) < 5:
            return 0
        rows = []
        for date, row in hist_df.iterrows():
            close = float(row.get('Close', row.get('Adj Close', 0)))
            if close <= 0 or pd.isna(close):
                continue
            rows.append((
                date.strftime('%Y-%m-%d') if hasattr(date, 'strftime') else str(date)[:10],
                ticker,
                float(row.get('Open',  close)),
                float(row.get('High',  close)),
                float(row.get('Low',   close)),
                close,
                close,   # adj_close (approximate -- yfinance auto_adjust=True)
                int(row.get('Volume', 0)),
            ))
        if rows:
            conn_inj.executemany(
                """INSERT OR REPLACE INTO prices
                   (date, ticker, open, high, low, close, adj_close, volume)
                   VALUES (?, ?, ?, ?, ?, ?, ?, ?)""",
                rows
            )
            conn_inj.commit()
        return len(rows)

    injected_ok = []
    failed = []

    # Try batch first (faster)
    try:
        raw = yf.download(
            downloadable,
            period="2y",
            auto_adjust=True,
            progress=True,
            group_by='ticker',
        )

        for ticker in downloadable:
            try:
                if len(downloadable) == 1:
                    hist = raw
                else:
                    hist = raw[ticker] if ticker in raw.columns.get_level_values(0) else pd.DataFrame()

                n = inject_ticker_prices(conn_fix, ticker, hist)
                if n >= 10:
                    injected_ok.append(ticker)
                    print(f"  [OK] {ticker}: {n} rows")
                else:
                    failed.append(ticker)
            except Exception:
                failed.append(ticker)

    except Exception as batch_err:
        print(f"  [!] Batch download failed ({batch_err}), trying individually...")
        failed = list(downloadable)

    # Individual fallback for anything that failed the batch
    if failed:
        print(f"\n[>>] Individual fallback for: {failed}")
        still_failed = []
        for ticker in failed:
            try:
                t_obj = yf.Ticker(ticker)
                hist = t_obj.history(period="2y", auto_adjust=True)
                n = inject_ticker_prices(conn_fix, ticker, hist)
                if n >= 10:
                    injected_ok.append(ticker)
                    print(f"  [OK] {ticker}: {n} rows (individual)")
                else:
                    still_failed.append(ticker)
                    print(f"  [SKIP] {ticker}: only {n} rows returned")
            except Exception as e2:
                still_failed.append(ticker)
                print(f"  [FAIL] {ticker}: {e2}")

        unexpected_failures = set(still_failed) - expected_failures
        if unexpected_failures:
            print(f"\n[!] Unexpected failures (investigate): {sorted(unexpected_failures)}")
        else:
            print(f"\n[OK] All failures are expected OTC/illiquid tickers.")

    print(f"\n[OK] Injection complete: {len(injected_ok)} tickers added.")

else:
    print("\n[OK] No missing tickers -- all prices present in DB.")

# ---------------------------------------------------------------------------
# Step 4c: Patch IDEQ holdings manually
# ---------------------------------------------------------------------------
# yfinance can't scrape IDEQ holdings (new ETF conversion May 2025).
# We inject top holdings from Lazard's Q4 2025 factsheet so the
# stock screener and NLP filings fetch don't skip IDEQ entirely.
print(f"\n[>>] Patching IDEQ holdings...")

ideq_top_holdings = [
    ('IDEQ', 'ASML',   'ASML Holding NV',              3.2),
    ('IDEQ', 'NVO',    'Novo Nordisk A/S ADR',          2.8),
    ('IDEQ', 'SAP',    'SAP SE ADR',                    2.5),
    ('IDEQ', 'NESN',   'Nestle SA',                     2.1),
    ('IDEQ', 'ROG',    'Roche Holding AG',               1.9),
    ('IDEQ', 'NOVN',   'Novartis AG ADR',               1.8),
    ('IDEQ', 'TM',     'Toyota Motor Corp ADR',         1.6),
    ('IDEQ', 'AIR',    'Airbus SE',                     1.5),
    ('IDEQ', 'SIEGY',  'Siemens AG ADR',                1.4),
    ('IDEQ', 'LVMUY',  'LVMH ADR',                      1.3),
    ('IDEQ', 'SAN',    'Banco Santander ADR',           1.2),
    ('IDEQ', 'BNPQY',  'BNP Paribas ADR',               1.1),
    ('IDEQ', 'IBDRY',  'Iberdrola ADR',                 1.0),
    ('IDEQ', 'RACE',   'Ferrari NV',                    0.9),
    ('IDEQ', 'BUD',    'AB InBev ADR',                  0.8),
    ('IDEQ', 'SONY',   'Sony Group ADR',                0.8),
    ('IDEQ', 'BCS',    'Barclays ADR',                  0.7),
    ('IDEQ', 'BP',     'BP plc ADR',                    0.7),
    ('IDEQ', 'RIO',    'Rio Tinto ADR',                 0.6),
    ('IDEQ', 'GSK',    'GSK plc ADR',                   0.6),
]

try:
    conn_fix.execute("""
        CREATE TABLE IF NOT EXISTS etf_holdings (
            etf_ticker     TEXT NOT NULL,
            holding_ticker TEXT NOT NULL,
            holding_name   TEXT DEFAULT '',
            weight_pct     REAL DEFAULT 0,
            updated_at     TEXT NOT NULL,
            PRIMARY KEY (etf_ticker, holding_ticker)
        )
    """)
    conn_fix.commit()

    now_str = dt.datetime.now().isoformat()
    conn_fix.executemany(
        """INSERT OR REPLACE INTO etf_holdings
           (etf_ticker, holding_ticker, holding_name, weight_pct, updated_at)
           VALUES (?, ?, ?, ?, ?)""",
        [(etf, holding, name, weight, now_str)
         for etf, holding, name, weight in ideq_top_holdings]
    )
    conn_fix.commit()
    print(f"[OK] IDEQ: {len(ideq_top_holdings)} holdings injected.")
except Exception as e:
    print(f"[!] IDEQ holdings patch failed (non-critical, screener will skip IDEQ): {e}")

# ---------------------------------------------------------------------------
# Step 4d: Final coverage check
# ---------------------------------------------------------------------------
print(f"\n=== Final Ticker Coverage ===")
loaded_final = set(
    row[0] for row in
    conn_fix.execute(
        "SELECT ticker FROM prices GROUP BY ticker HAVING COUNT(*) >= 100"
    ).fetchall()
)

# Core tickers the optimizer absolutely needs
critical = set(config['tickers']['sector_etfs']) | {
    'SPY', 'BIL', 'SGOV', 'VGK', 'VWO', 'INDA', 'EWZ',
    'FCOM', 'FDIS', 'FHLC', 'FIDU', 'IDEQ',
    'HUBS', 'XYZ', 'BMRN', 'FIS', 'NVDA',
}
missing_critical = critical - loaded_final
if missing_critical:
    print(f"[!!] CRITICAL tickers still missing -- optimizer may fail: {sorted(missing_critical)}")
else:
    print(f"[OK] All critical tickers present in DB.")

print(f"     Total tickers with sufficient history: {len(loaded_final)}")
conn_fix.close()

[>>] Tickers needed:    171
[OK] Already in DB:     153
[!]  Missing / thin:    20
     ['FCOM', 'FDIS', 'FENY', 'FHLC', 'FIDU', 'FLBR', 'FLCH', 'FLEU', 'FLIN', 'FLJP', 'FLKR', 'FLTW', 'FMAT', 'FNCL', 'FREL', 'FSTA', 'FTEC', 'FUTY', 'IEMG', 'SGOV']

[>>] Downloading 20 missing tickers via yfinance...


[*********************100%***********************]  20 of 20 completed


  [OK] FCOM: 502 rows
  [OK] FDIS: 502 rows
  [OK] FENY: 502 rows
  [OK] FHLC: 502 rows
  [OK] FIDU: 502 rows
  [OK] FLBR: 502 rows
  [OK] FLCH: 502 rows
  [OK] FLEU: 502 rows
  [OK] FLIN: 502 rows
  [OK] FLJP: 502 rows
  [OK] FLKR: 502 rows
  [OK] FLTW: 502 rows
  [OK] FMAT: 502 rows
  [OK] FNCL: 502 rows
  [OK] FREL: 502 rows
  [OK] FSTA: 502 rows
  [OK] FTEC: 502 rows
  [OK] FUTY: 502 rows
  [OK] IEMG: 502 rows
  [OK] SGOV: 502 rows

[OK] Injection complete: 20 tickers added.

[>>] Patching IDEQ holdings...
[OK] IDEQ: 20 holdings injected.

=== Final Ticker Coverage ===
[OK] All critical tickers present in DB.
     Total tickers with sufficient history: 173


## 5. Detect Current Regime

Detects the current market regime (Offense / Defense / Panic) using the wedge-volume percentile and VIX/RV ratio.

In [5]:
import regime_detector
importlib.reload(regime_detector)
patch_db_path(regime_detector, DB_PATH)

regime_result = regime_detector.run_regime_detection(config)

print("=" * 60)
print("CURRENT REGIME")
print("=" * 60)
print(f"  Regime:              {regime_result.get('dominant_regime', 'N/A')}")
print(f"  Wedge Volume %:      {regime_result.get('wedge_volume_percentile', 'N/A')}")
print(f"  Fast Shock Risk:     {regime_result.get('fast_shock_risk', 'N/A')}")
print(f"  VIX/RV Ratio:        {regime_result.get('vix_rv_ratio', 'N/A')}")
print(f"  Confirmed:           {regime_result.get('regime_confirmed', 'N/A')}")
print(f"  Consecutive Days:    {regime_result.get('consecutive_days_in_regime', 'N/A')}")

  regime_detector.DB_PATH -> /content/sector-rotation-system/rotation_system.db
CURRENT REGIME
  Regime:              offense
  Wedge Volume %:      84.58
  Fast Shock Risk:     high
  VIX/RV Ratio:        1.8294
  Confirmed:           True
  Consecutive Days:    40


## 6. Stock Screener — Top Picks

Scores all watchlist stocks across biotech, AI software, defense, materials, semiconductors, energy transition, and fintech. Generates entry/exit signals. Output feeds directly into the confirmation step below.

In [6]:
import stock_screener
importlib.reload(stock_screener)
patch_db_path(stock_screener, DB_PATH)

regime = regime_result.get('dominant_regime', 'offense')
screener_result = stock_screener.run_stock_screener(cfg=config, regime=regime)

print("=" * 60)
print("THEMATIC WATCHLIST SCORES")
print("=" * 60)

watchlists = screener_result.get('watchlist_scores', {})
for name, data in watchlists.items():
    if data:
        print(f"\n--- {name.upper()} ---")
        df = pd.DataFrame(data) if isinstance(data, list) else data
        if not df.empty:
            show_cols = [c for c in ['ticker', 'composite_score', 'momentum_score',
                                      'quality_score', 'value_score', 'size_score',
                                      'valuation_label', 'signal'] if c in df.columns]
            print(df[show_cols].head(10).to_string(index=False))

signals_out = screener_result.get('signals', {})
if signals_out:
    print(f"\n{'='*60}")
    print("ENTRY / EXIT SIGNALS")
    print(f"{'='*60}")
    for sig_type in ['entry', 'exit']:
        sigs = signals_out.get(sig_type, [])
        print(f"\n--- {sig_type.upper()} {'(none)' if not sigs else ''} ---")
        for s in sigs[:10]:
            print(f"    {s.get('ticker','?')} ({s.get('watchlist','')}): {s.get('reason','')}")

print(f"\n{'='*60}")
print("SECTOR TOP PICKS")
print(f"{'='*60}")
stp_data = screener_result.get('sector_top_picks', [])
if stp_data:
    stp_df = pd.DataFrame(stp_data)
    show_cols = [c for c in ['ticker','etf','composite_score','momentum_rank',
                               'quality_score','value_score','valuation_label'] if c in stp_df.columns]
    print(stp_df[show_cols].to_string(index=False))
else:
    print("  (no sector top picks -- no overweight sectors identified)")

_screener_json = Path(os.getcwd()) / 'screener_output.json'
if _screener_json.exists():
    print(f"\n[OK] screener_output.json written ({_screener_json.stat().st_size:,} bytes)")
else:
    print("\n[!] screener_output.json NOT found -- satellite picks will not be injected.")

  stock_screener.DB_PATH -> /content/sector-rotation-system/rotation_system.db


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: HES"}}}
ERROR:yfinance:
2 Failed downloads:
ERROR:yfinance:['PXD', 'HES']: YFPricesMissingError('possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")')
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: HES"}}}


THEMATIC WATCHLIST SCORES

--- BIOTECH ---
ticker  composite_score  quality_score valuation_label
  TERN           3.3490         0.0579 FUNDAMENTAL_BUY
  HALO           0.6807         0.8900 FUNDAMENTAL_BUY
  GPCR           0.6694         0.0530 FUNDAMENTAL_BUY
  EXEL           0.6157         0.8651 FUNDAMENTAL_BUY
  IONS           0.5727         0.0513 FUNDAMENTAL_BUY
  NBIX           0.5674         0.6468 FUNDAMENTAL_BUY
  ALNY           0.4791         0.7202 FUNDAMENTAL_BUY
  BMRN           0.4626         0.6342 FUNDAMENTAL_BUY
  VKTX           0.0955         0.0275 FUNDAMENTAL_BUY
  MRNA           0.0858         0.0067 FUNDAMENTAL_BUY

--- AI_SOFTWARE ---
ticker  composite_score  quality_score  value_score valuation_label
  DOCN           0.4812         0.6987       0.2132   MOMENTUM_ONLY
  FSLY           0.4204         0.3827       0.0078           AVOID
  PLTR           0.4033         0.6266       0.0003     EXCEEDS_CAP
   MDB           0.4000         0.4428       0.0861        

## 7. Review & Confirm Screener Picks

**Review before proceeding.** Confirm which stocks enter the Roth satellite sleeve and which ETFs are in play. Edit `force_include` / `force_exclude` to override screener decisions, then run the optimizer.

In [7]:
# -----------------------------------------------------------------------
# Review screener output before the optimizer runs.
# Add tickers to force_include to guarantee they enter the Roth satellite sleeve.
# Add tickers to force_exclude to block them from the optimizer entirely.
# Leave both as [] to accept the screener output as-is.
# -----------------------------------------------------------------------

force_include = []   # e.g. ['NBIX', 'HALO'] -- force into Roth satellite
force_exclude = []   # e.g. ['ARKK', 'VKTX'] -- block from optimizer

# Show current screener picks
print("=" * 70)
print("SCREENER PICKS ENTERING OPTIMIZER")
print("=" * 70)

satellite_picks = screener_result.get('satellite_picks', [])
entry_signals   = screener_result.get('signals', {}).get('entry', [])

# Merge entry signals + existing Roth holdings as confirmed picks
confirmed_picks = list({
    s.get('ticker') for s in entry_signals if s.get('ticker')
} | {'HUBS', 'XYZ', 'BMRN', 'FIS', 'NVDA'})  # always keep current Roth holdings

# Apply overrides
for t in force_exclude:
    if t in confirmed_picks:
        confirmed_picks.remove(t)
        print(f"  [EXCLUDED] {t}")
for t in force_include:
    if t not in confirmed_picks:
        confirmed_picks.append(t)
        print(f"  [FORCED IN] {t}")

print(f"\n  Roth satellite stocks ({len(confirmed_picks)}):")
for t in sorted(confirmed_picks):
    reason = next((s.get('reason','') for s in entry_signals if s.get('ticker') == t), 'current holding')
    print(f"    {t:<8} {reason}")

# Inject confirmed picks back into screener_result so optimizer reads them
screener_result['confirmed_satellite_picks'] = confirmed_picks

print(f"\n  ETF universe: {len(config['tickers']['sector_etfs'])} SPDR XL sectors")
print(f"  Geographic ETFs in play: {config['tickers']['geographic_etfs'][:5]} ...")
print(f"\n[OK] Confirmed. Run next cell to optimize.")

SCREENER PICKS ENTERING OPTIMIZER

  Roth satellite stocks (18):
    AMAT     Top quartile score (0.625), valuation=FUNDAMENTAL_BUY, regime=offense
    BMRN     current holding
    EXEL     Top quartile score (0.616), valuation=FUNDAMENTAL_BUY, regime=offense
    FIS      current holding
    FSLR     Top quartile score (0.655), valuation=FUNDAMENTAL_BUY, regime=offense
    GPCR     Top quartile score (0.669), valuation=FUNDAMENTAL_BUY, regime=offense
    HALO     Top quartile score (0.681), valuation=FUNDAMENTAL_BUY, regime=offense
    HII      Top quartile score (0.611), valuation=FUNDAMENTAL_BUY, regime=offense
    HUBS     current holding
    LDOS     Top quartile score (0.557), valuation=FUNDAMENTAL_BUY, regime=offense
    NEM      Top quartile score (0.942), valuation=FUNDAMENTAL_BUY, regime=offense
    NU       Top quartile score (0.469), valuation=FUNDAMENTAL_BUY, regime=offense
    NVDA     current holding
    SEDG     Top quartile score (0.681), valuation=FUNDAMENTAL_BUY, regi

## 8. CVaR Optimization

Runs the full CVaR optimizer using the confirmed screener picks from the previous cell. Uses the ETF auto-selector to substitute cheaper funds (e.g. FCOM over XLC, SGOV over BIL). Includes a **current-holdings fallback** if the optimizer returns zero positions.

In [8]:
import portfolio_optimizer
importlib.reload(portfolio_optimizer)
patch_db_path(portfolio_optimizer, DB_PATH)

try:
    import etf_selector
    importlib.reload(etf_selector)
    portfolio_optimizer._get_etf_selections = etf_selector.get_selected_tickers
    portfolio_optimizer._get_selector_asset_class_map = etf_selector.get_ticker_asset_class_map
    portfolio_optimizer._ETF_SELECTOR_AVAILABLE = True
    print("  [OK] ETF auto-selector loaded and active")
except ImportError:
    print("  [!] ETF selector not available -- using legacy SPDR tickers")

opt_result = portfolio_optimizer.run_portfolio_optimization(cfg=config, regime=regime)

total     = config['portfolio']['total_value']
taxable_val = config['portfolio']['accounts']['taxable']['value']
roth_val    = config['portfolio']['accounts']['roth_ira']['value']

print("=" * 80)
print(f"CVaR-OPTIMIZED ALLOCATION (Regime: {regime.upper()})")
print(f"Portfolio: ${total:,.0f} = ${taxable_val:,.0f} Taxable + ${roth_val:,.0f} Roth IRA")
print("=" * 80)

positions = opt_result.get('positions', {})

if not positions:
    # -----------------------------------------------------------------------
    # FALLBACK: optimizer returned empty -- inject current holdings directly
    # so the rest of the notebook (backtest, review, download) still works.
    # This happens when the sub-sector allocation logic can't find enough
    # tickers in an asset class band. The fallback uses your actual weights.
    # -----------------------------------------------------------------------
    print("\n[!] Optimizer returned no positions. Applying current-holdings fallback...\n")

    FALLBACK_POSITIONS = {
        # Taxable -- Fidelity MSCI sector ETFs + IDEQ
        'FCOM': {'pct': 5.49,  'account': 'taxable',  'taxable_dollars': 12967, 'roth_dollars': 0,     'total_dollars': 12967, 'reason': 'current holding -- taxable'},
        'FDIS': {'pct': 3.03,  'account': 'taxable',  'taxable_dollars': 7155,  'roth_dollars': 0,     'total_dollars': 7155,  'reason': 'current holding -- taxable'},
        'FHLC': {'pct': 7.49,  'account': 'taxable',  'taxable_dollars': 17665, 'roth_dollars': 0,     'total_dollars': 17665, 'reason': 'current holding -- taxable'},
        'FIDU': {'pct': 9.54,  'account': 'taxable',  'taxable_dollars': 22513, 'roth_dollars': 0,     'total_dollars': 22513, 'reason': 'current holding -- taxable'},
        'IDEQ': {'pct': 16.34, 'account': 'taxable',  'taxable_dollars': 38554, 'roth_dollars': 0,     'total_dollars': 38554, 'reason': 'current holding -- taxable (conviction intl)'},
        # Roth IRA -- individual stocks
        'HUBS': {'pct': 5.47,  'account': 'roth_ira', 'taxable_dollars': 0,     'roth_dollars': 12902, 'total_dollars': 12902, 'reason': 'current holding -- roth satellite stock'},
        'XYZ':  {'pct': 4.61,  'account': 'roth_ira', 'taxable_dollars': 0,     'roth_dollars': 10867, 'total_dollars': 10867, 'reason': 'current holding -- roth satellite stock'},
        'BMRN': {'pct': 4.28,  'account': 'roth_ira', 'taxable_dollars': 0,     'roth_dollars': 10108, 'total_dollars': 10108, 'reason': 'current holding -- roth satellite stock'},
        'FIS':  {'pct': 4.15,  'account': 'roth_ira', 'taxable_dollars': 0,     'roth_dollars': 9783,  'total_dollars': 9783,  'reason': 'current holding -- roth satellite stock'},
        'NVDA': {'pct': 1.08,  'account': 'roth_ira', 'taxable_dollars': 0,     'roth_dollars': 2547,  'total_dollars': 2547,  'reason': 'current holding -- roth satellite stock'},
    }

    opt_result['positions'] = FALLBACK_POSITIONS
    positions = FALLBACK_POSITIONS
    print("[OK] Fallback positions loaded. These reflect your actual current holdings.")
    print("     To get optimizer-generated targets, investigate portfolio_optimizer.py")
    print("     error: 'All weights are zero after sub-sector allocation'.\n")

etf_positions   = {t: i for t, i in positions.items() if 'satellite stock' not in i.get('reason', '')}
stock_positions = {t: i for t, i in positions.items() if 'satellite stock'     in i.get('reason', '')}

print(f"\n{'Ticker':<8} {'Type':<6} {'Weight':>7} {'Total $':>10} {'Taxable $':>10} {'Roth $':>10}  Account")
print("-" * 80)
for ticker, info in sorted(etf_positions.items(), key=lambda x: -x[1].get('pct', 0)):
    pct   = info.get('pct', 0)
    tot_d = info.get('total_dollars', pct / 100.0 * total)
    tax_d = info.get('taxable_dollars', 0)
    rth_d = info.get('roth_dollars', 0)
    acct  = info.get('account', 'N/A')
    print(f"{ticker:<8} {'ETF':<6} {pct:>6.1f}% {tot_d:>10,.0f} {tax_d:>10,.0f} {rth_d:>10,.0f}  {acct}")

if stock_positions:
    print(f"\n  -- Roth Satellite Stocks --")
    for ticker, info in sorted(stock_positions.items(), key=lambda x: -x[1].get('pct', 0)):
        pct   = info.get('pct', 0)
        tot_d = info.get('total_dollars', pct / 100.0 * total)
        tax_d = info.get('taxable_dollars', 0)
        rth_d = info.get('roth_dollars', 0)
        acct  = info.get('account', 'N/A')
        print(f"{ticker:<8} {'STOCK':<6} {pct:>6.1f}% {tot_d:>10,.0f} {tax_d:>10,.0f} {rth_d:>10,.0f}  {acct}")

tax_total  = sum(i.get('taxable_dollars', 0) for i in positions.values())
roth_total = sum(i.get('roth_dollars', 0)   for i in positions.values())
print("-" * 80)
print(f"{'TOTAL':<8} {'':6} {'100%':>7} {total:>10,.0f} {tax_total:>10,.0f} {roth_total:>10,.0f}")
print(f"\nAccount utilization: Taxable {tax_total/taxable_val:.0%} | Roth {roth_total/roth_val:.0%}")

tickers_out = set(positions.keys())
fidelity_check = tickers_out & {'FTEC','FHLC','FNCL','FIDU','FDIS','FSTA','FENY','FMAT','FUTY','FREL','FCOM'}
spdr_check     = tickers_out & {'XLK','XLV','XLF','XLI','XLP','XLU','XLC','XLY','XLE','XLB','XLRE'}
print(f"\n-- ETF Auto-Selector --")
if fidelity_check: print(f"  [OK] Fidelity MSCI active: {sorted(fidelity_check)}")
if 'SGOV' in tickers_out: print(f"  [OK] Cash: SGOV")
if spdr_check: print(f"  [!]  SPDR still present (not substituted): {sorted(spdr_check)}")

  portfolio_optimizer.DB_PATH -> /content/sector-rotation-system/rotation_system.db
  [OK] ETF auto-selector loaded and active
CVaR-OPTIMIZED ALLOCATION (Regime: OFFENSE)
Portfolio: $235,970 = $98,913 Taxable + $46,207 Roth IRA

Ticker   Type    Weight    Total $  Taxable $     Roth $  Account
--------------------------------------------------------------------------------
FENY     ETF      18.2%     42,983     42,983          0  taxable
VGK      ETF      15.0%     35,396     35,396          0  taxable
FIDU     ETF       9.9%     23,277          0     23,277  roth_ira
FHLC     ETF       9.3%     22,041     20,535      1,506  split
FCOM     ETF       8.6%     20,339          0     12,183  roth_ira
VNQ      ETF       6.7%     15,799          0          0  taxable
FDIS     ETF       5.7%     13,397          0          0  roth_ira
FTEC     ETF       4.5%     10,607          0          0  roth_ira
FUTY     ETF       4.5%     10,522          0          0  taxable
FNCL     ETF       4.0%     

## 9. Walk-Forward Backtest

Simulates the system over the historical period, rebalancing monthly with regime-conditional weights. Includes an **equal-weight fallback** if the optimizer fails all rebalance dates.

In [9]:
import numpy as np
import json as json_module

import portfolio_optimizer
importlib.reload(portfolio_optimizer)
patch_db_path(portfolio_optimizer, DB_PATH)

total = config['portfolio']['total_value']

conn = sqlite3.connect(str(DB_PATH))

opt_universe = (
    config['tickers']['sector_etfs']
    + config['tickers']['geographic_etfs']
    + ['BIL']
)
opt_universe = list(dict.fromkeys(opt_universe))

# Remove tickers with no price history
_conn_check = sqlite3.connect(str(DB_PATH))
available_tickers = set(
    row[0] for row in
    _conn_check.execute(
        "SELECT ticker FROM prices GROUP BY ticker HAVING COUNT(*) >= 100"
    ).fetchall()
)
_conn_check.close()
opt_universe = [t for t in opt_universe if t in available_tickers]
print(f"Optimization universe after availability filter: {len(opt_universe)} tickers")

placeholders = ','.join(['?'] * len(opt_universe))
etf_prices = pd.read_sql(
    f"SELECT date, ticker, adj_close FROM prices WHERE ticker IN ({placeholders}) ORDER BY date",
    conn, params=opt_universe, parse_dates=['date']
)

spy = pd.read_sql(
    "SELECT date, adj_close FROM prices WHERE ticker='SPY' ORDER BY date",
    conn, parse_dates=['date']
).set_index('date')
spy_returns = spy['adj_close'].pct_change().dropna()

signals_df = pd.read_sql(
    "SELECT date, signal_data FROM signals WHERE signal_type='regime_state' ORDER BY date",
    conn, parse_dates=['date']
)
if len(signals_df) > 0:
    signals_df['regime'] = signals_df['signal_data'].apply(
        lambda x: json_module.loads(x).get('dominant_regime', 'offense')
        if isinstance(x, str) else 'offense'
    )
    regime_series = signals_df.set_index('date')['regime']
else:
    regime_series = pd.Series(dtype=str, name='regime')

prices_pivot   = etf_prices.pivot(index='date', columns='ticker', values='adj_close')
etf_daily_rets = prices_pivot.pct_change()

common_idx    = spy_returns.index.intersection(etf_daily_rets.dropna(how='all').index)
spy_returns   = spy_returns.loc[common_idx]
etf_daily_rets = etf_daily_rets.loc[common_idx]

print(f"SPY returns: {len(spy_returns)} days | ETF universe: {len(opt_universe)} tickers")
print(f"Regime signals: {len(regime_series)} rows")

monthly_first = {}
for date in etf_daily_rets.index:
    key = (date.year, date.month)
    if key not in monthly_first:
        monthly_first[key] = date
rebalance_dates = sorted(monthly_first.values())
print(f"Rebalance dates: {len(rebalance_dates)} months")

print("Running optimizer for each rebalance date...")
weights_schedule = {}
optimizer_failures = 0

for rb_date in rebalance_dates:
    past = regime_series[regime_series.index <= rb_date]
    regime_at = past.iloc[-1] if len(past) > 0 else 'offense'
    try:
        opt_res = portfolio_optimizer.run_portfolio_optimization(
            conn=conn, cfg=config, regime=regime_at
        )
        positions_rb = opt_res.get('positions', {})
        if positions_rb:
            raw_w = {t: info.get('pct', 0) / 100.0 for t, info in positions_rb.items()}
            available = {t: w for t, w in raw_w.items() if t in etf_daily_rets.columns}
            total_w = sum(available.values())
            if total_w > 0:
                weights_schedule[rb_date] = {t: w / total_w for t, w in available.items()}
        else:
            optimizer_failures += 1
    except Exception as e:
        optimizer_failures += 1

print(f"Optimizer: {len(weights_schedule)} successful / {optimizer_failures} failed rebalance dates")

# If optimizer failed for all dates, use equal-weight fallback for backtest
if len(weights_schedule) == 0:
    print("\n[!] Optimizer failed all rebalance dates.")
    print("    Using equal-weight fallback across available sector ETFs for backtest illustration.")
    available_xl = [t for t in config['tickers']['sector_etfs'] if t in etf_daily_rets.columns]
    eq_weight = {t: 1.0 / len(available_xl) for t in available_xl} if available_xl else {}
    for rb_date in rebalance_dates:
        weights_schedule[rb_date] = eq_weight
    print(f"    Equal-weight portfolio: {len(available_xl)} XL sector ETFs")

strategy_rets = []
current_weights = {}
for date in etf_daily_rets.index:
    if date in weights_schedule:
        current_weights = weights_schedule[date]
    if current_weights:
        row = etf_daily_rets.loc[date]
        port_ret = sum(
            w * (row[t] if pd.notna(row.get(t, float('nan'))) else 0.0)
            for t, w in current_weights.items()
        )
    else:
        port_ret = 0.0
    strategy_rets.append(port_ret)

strategy_returns = pd.Series(strategy_rets, index=etf_daily_rets.index, name='strategy_ret')
conn.close()

print(f"Strategy returns computed: {len(strategy_returns)} days")

if len(spy_returns) > 0 and len(strategy_returns) > 0:
    merged = spy_returns.to_frame('spy_ret').join(strategy_returns, how='inner')
    merged['regime'] = regime_series.reindex(merged.index, method='ffill').fillna('offense')
    merged['spy_cum']      = (1 + merged['spy_ret']).cumprod()
    merged['strategy_cum'] = (1 + merged['strategy_ret']).cumprod()

    days       = len(merged)
    ann_factor = 252 / days if days > 0 else 1

    spy_total   = merged['spy_cum'].iloc[-1] - 1
    strat_total = merged['strategy_cum'].iloc[-1] - 1
    spy_ann     = (1 + spy_total) ** ann_factor - 1
    strat_ann   = (1 + strat_total) ** ann_factor - 1
    spy_vol     = merged['spy_ret'].std() * np.sqrt(252)
    strat_vol   = merged['strategy_ret'].std() * np.sqrt(252)
    spy_sharpe  = spy_ann / spy_vol if spy_vol > 0 else 0
    strat_sharpe = strat_ann / strat_vol if strat_vol > 0 else 0

    def max_dd(cum_series):
        peak = cum_series.cummax()
        return ((cum_series - peak) / peak).min()

    mclean_decay = config['factor_model']['mclean_pontiff_decay']
    alpha_raw    = strat_ann - spy_ann
    alpha_adj    = alpha_raw * mclean_decay

    print("=" * 70)
    print("WALK-FORWARD BACKTEST RESULTS")
    print("=" * 70)
    print(f"Period: {merged.index[0].strftime('%Y-%m-%d')} to {merged.index[-1].strftime('%Y-%m-%d')} ({days} days)")
    print()
    print(f"{'Metric':<30} {'SPY (B&H)':>15} {'Strategy':>15}")
    print("-" * 60)
    print(f"{'Total Return':<30} {spy_total:>14.1%} {strat_total:>14.1%}")
    print(f"{'Annualized Return':<30} {spy_ann:>14.1%} {strat_ann:>14.1%}")
    print(f"{'Annualized Volatility':<30} {spy_vol:>14.1%} {strat_vol:>14.1%}")
    print(f"{'Sharpe Ratio':<30} {spy_sharpe:>14.2f} {strat_sharpe:>14.2f}")
    print(f"{'Max Drawdown':<30} {max_dd(merged['spy_cum']):>14.1%} {max_dd(merged['strategy_cum']):>14.1%}")
    print(f"{'Dollar Gain (${:,.0f} port.)':<30} {spy_total * total:>13,.0f} {strat_total * total:>13,.0f}".format(total))
    print()
    print(f"Raw Alpha vs SPY:       {alpha_raw:>+.2%}")
    print(f"Adjusted Alpha (x{mclean_decay}):  {alpha_adj:>+.2%}")
    print(f"  {config['backtest']['mclean_pontiff_label']}")
    print(f"\nRegime distribution:")
    for r, ct in merged['regime'].value_counts().items():
        print(f"  {r}: {ct} days ({ct/days:.0%})")

  portfolio_optimizer.DB_PATH -> /content/sector-rotation-system/rotation_system.db
Optimization universe after availability filter: 22 tickers
SPY returns: 500 days | ETF universe: 22 tickers
Regime signals: 248 rows
Rebalance dates: 25 months
Running optimizer for each rebalance date...
Optimizer: 25 successful / 0 failed rebalance dates
Strategy returns computed: 500 days
WALK-FORWARD BACKTEST RESULTS
Period: 2024-03-07 to 2026-03-05 (500 days)

Metric                               SPY (B&H)        Strategy
------------------------------------------------------------
Total Return                            37.0%          36.7%
Annualized Return                       17.2%          17.1%
Annualized Volatility                   16.4%          14.9%
Sharpe Ratio                             1.05           1.15
Max Drawdown                           -18.8%         -14.3%
Dollar Gain ($235,970 port.)          87,214        86,537

Raw Alpha vs SPY:       -0.12%
Adjusted Alpha (x0.74):  -0

## 10. Full Monitor (Executive Summary)

Runs the full monitor pipeline and displays the executive summary. `--no-deliver` skips Telegram/email/Sheets.

In [10]:
os.system(f'python monitor.py --no-deliver --db "{DB_PATH}" 2>&1 | head -120')

0

## 11. Review & Confirm Final Allocation

**Review before downloading.** Edit `overrides` to adjust any weights — they will be re-normalized to 100% automatically. Leave `overrides = {}` to accept the optimizer output as-is.

In [11]:
import json

positions = opt_result.get('positions', {})
total       = config['portfolio']['total_value']
taxable_val = config['portfolio']['accounts']['taxable']['value']
roth_val    = config['portfolio']['accounts']['roth_ira']['value']

# -----------------------------------------------------------------------
# Edit overrides below to adjust weights before downloading.
# Example: overrides = {'FCOM': 8.0, 'IDEQ': 12.0, 'SGOV': 0.0}
# Weights are in PERCENT and will be re-normalized to 100%.
# Leave as {} to accept the current output as-is.
# -----------------------------------------------------------------------
overrides = {}

current_weights = {t: info['pct'] for t, info in positions.items()}

if overrides:
    print("[>>] Applying manual overrides...")
    for ticker, new_pct in overrides.items():
        old_pct = current_weights.get(ticker, 0)
        current_weights[ticker] = new_pct
        print(f"  {ticker}: {old_pct:.1f}% -> {new_pct:.1f}%")
    current_weights = {t: w for t, w in current_weights.items() if w > 0}
    total_pct = sum(current_weights.values())
    if total_pct > 0 and abs(total_pct - 100.0) > 0.1:
        scale = 100.0 / total_pct
        current_weights = {t: round(w * scale, 2) for t, w in current_weights.items()}
        print(f"  Re-normalized to 100.0%")
    weight_fracs = {t: w / 100.0 for t, w in current_weights.items()}
    importlib.reload(portfolio_optimizer)
    positions = portfolio_optimizer.allocate_dollars(weight_fracs, config)
    opt_result['positions'] = positions
    print("  [OK] Dollar allocation recalculated.\n")
else:
    print("No overrides -- using current output as-is.\n")

etf_pos   = {t: i for t, i in positions.items() if 'satellite stock' not in i.get('reason', '')}
stock_pos = {t: i for t, i in positions.items() if 'satellite stock'     in i.get('reason', '')}

print("=" * 100)
print(f"  FINAL PORTFOLIO -- {regime.upper()} REGIME")
print(f"  ${total:,.0f} = ${taxable_val:,.0f} Taxable + ${roth_val:,.0f} Roth IRA")
print("=" * 100)

header = f"{'#':<3} {'Ticker':<8} {'Type':<6} {'Weight':>7} {'Total $':>10} {'Taxable $':>10} {'Roth $':>10}  {'Account':<10} Reason"
divider = "-" * 110
print(f"\n-- ETF Core Positions --")
print(header)
print(divider)

tax_total = roth_total_ = 0
for i, (ticker, info) in enumerate(sorted(etf_pos.items(), key=lambda x: -x[1].get('pct', 0)), 1):
    pct   = info.get('pct', 0)
    tot_d = info.get('total_dollars', 0)
    tax_d = info.get('taxable_dollars', 0)
    rth_d = info.get('roth_dollars', 0)
    acct  = info.get('account', 'N/A')
    tax_total += tax_d; roth_total_ += rth_d
    print(f"{i:<3} {ticker:<8} {'ETF':<6} {pct:>6.1f}% ${tot_d:>9,.0f} ${tax_d:>9,.0f} ${rth_d:>9,.0f}  {acct:<10} {info.get('reason','')}")

if stock_pos:
    print(f"\n-- Roth IRA Satellite Stocks --")
    print(header)
    print(divider)
    for j, (ticker, info) in enumerate(sorted(stock_pos.items(), key=lambda x: -x[1].get('pct', 0)), len(etf_pos) + 1):
        pct   = info.get('pct', 0)
        tot_d = info.get('total_dollars', 0)
        tax_d = info.get('taxable_dollars', 0)
        rth_d = info.get('roth_dollars', 0)
        acct  = info.get('account', 'N/A')
        tax_total += tax_d; roth_total_ += rth_d
        print(f"{j:<3} {ticker:<8} {'STOCK':<6} {pct:>6.1f}% ${tot_d:>9,.0f} ${tax_d:>9,.0f} ${rth_d:>9,.0f}  {acct:<10} {info.get('reason','')}")

print(divider)
print(f"{'':3} {'TOTAL':<8} {'':6} {'100.0':>6}% ${tax_total+roth_total_:>9,.0f} ${tax_total:>9,.0f} ${roth_total_:>9,.0f}")
print(f"\n  Taxable: ${tax_total:,.0f} / ${taxable_val:,.0f} ({tax_total/taxable_val:.0%})")
print(f"  Roth:    ${roth_total_:,.0f} / ${roth_val:,.0f} ({roth_total_/roth_val:.0%})")
print(f"\n{'='*80}")
print("  [OK] If this looks correct, run Cell 11 to download.")
print("  [>>] To change weights, edit overrides dict above and re-run.")
print(f"{'='*80}")

No overrides -- using current output as-is.

  FINAL PORTFOLIO -- OFFENSE REGIME
  $235,970 = $98,913 Taxable + $46,207 Roth IRA

-- ETF Core Positions --
#   Ticker   Type    Weight    Total $  Taxable $     Roth $  Account    Reason
--------------------------------------------------------------------------------------------------------------
1   FENY     ETF      18.2% $   42,983 $   42,983 $        0  taxable    Taxable: geographic/cash/energy (foreign tax credit / low turnover)
2   VGK      ETF      15.0% $   35,396 $   35,396 $        0  taxable    Taxable: geographic/cash/energy (foreign tax credit / low turnover)
3   FIDU     ETF       9.9% $   23,277 $        0 $   23,277  roth_ira   Roth: taxable capacity full
4   FHLC     ETF       9.3% $   22,041 $   20,535 $    1,506  split      Split: capacity constrained ($20535 Taxable + $1506 Roth)
5   FCOM     ETF       8.6% $   20,339 $        0 $   12,183  roth_ira   Roth: last available capacity
6   VNQ      ETF       6.7% $   15,79

## 12. Download Database

Saves the reviewed allocation to `current_allocation.json` and downloads the populated database for use with the Streamlit dashboard.

In [12]:
from google.colab import files

alloc_path = DB_PATH.parent / 'current_allocation.json'
with open(alloc_path, 'w') as f:
    json.dump(opt_result, f, indent=2, default=str)
print(f"Allocation saved to {alloc_path}")

files.download(str(DB_PATH))
print("Database downloaded. Place it in the repo root for the dashboard.")

Allocation saved to /content/sector-rotation-system/current_allocation.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Database downloaded. Place it in the repo root for the dashboard.
